Title: Data Exploration
Author: Kolbe Sussman


In [ ]:
import pandas as pd
import networkx as nx
from itertools import combinations
from collections import Counter
import ast
import matplotlib.pyplot as plt


In [4]:
df = pd.read_csv("../data/processed/umich_works_cleaned.csv")
df.head()

,id,doi,title,authorships,topics,primary_topic,cited_by_count,publication_year,related_works,concepts,authorships_parsed,author_ids,author_names,institutions,raw_affiliations,topics_parsed,topic_ids,display_names,topic_scores
0,https://openalex.org/W4235678817,https://doi.org/10.1177/002224378101800104,Evaluating Structural Equation Models with Uno...,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10467', 'displa...","{'id': 'https://openalex.org/T10467', 'display...",64467,1981,"['https://openalex.org/W2614563012', 'https://...","[{'id': 'https://openalex.org/C2780695315', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5047556013', 'https://...","['Claes Fornell', 'David F. Larcker']",['University of Michigan–Ann Arbor'],"['The University of Michigan', 'Northwestern U...","[{'id': 'https://openalex.org/T10467', 'displa...","['https://openalex.org/T10467', 'https://opena...","['Psychometric Methodologies and Testing', 'Mu...","[0.9793999791145325, 0.9490000009536743, 0.941..."
1,https://openalex.org/W1791587663,https://doi.org/10.2307/249008,"Perceived Usefulness, Perceived Ease of Use, a...","[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10068', 'displa...","{'id': 'https://openalex.org/T10068', 'display...",61935,1989,"['https://openalex.org/W4389670110', 'https://...","[{'id': 'https://openalex.org/C2776185967', 'w...","[{'author_position': 'first', 'author': {'id':...",['https://openalex.org/A5103061328'],['Fred D. Davis'],['University of Michigan–Ann Arbor'],['Computer and Information Systems'],"[{'id': 'https://openalex.org/T10068', 'displa...","['https://openalex.org/T10068', 'https://opena...","['Technology Adoption and User Behaviour', 'Kn...","[0.9998000264167786, 0.9921000003814697, 0.985..."
2,https://openalex.org/W1987258130,https://doi.org/10.2307/3151312,Evaluating Structural Equation Models with Uno...,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10467', 'displa...","{'id': 'https://openalex.org/T10467', 'display...",60116,1981,"['https://openalex.org/W2614563012', 'https://...","[{'id': 'https://openalex.org/C2780695315', 'w...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5047556013', 'https://...","['Claes Fornell', 'David F. Larcker']","['University of Michigan–Ann Arbor', 'Northwes...","['Northwestern University****', 'U. Michigan#T...","[{'id': 'https://openalex.org/T10467', 'displa...","['https://openalex.org/T10467', 'https://opena...","['Psychometric Methodologies and Testing', 'Ad...","[0.9904999732971191, 0.9409000277519226]"
3,https://openalex.org/W2097117768,https://doi.org/10.1109/cvpr.2015.7298594,Going deeper with convolutions,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10036', 'displa...","{'id': 'https://openalex.org/T10036', 'display...",46263,2015,"['https://openalex.org/W2364252372', 'https://...","[{'id': 'https://openalex.org/C41008148', 'wik...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/A5002183320', 'https://...","['Christian Szegedy', 'Wei Liu', 'Yangqing Jia...","['Magic Leap (United States)', 'Google (United...","['Google', 'IGoogle Inc', 'University of North...","[{'id': 'https://openalex.org/T10036', 'displa...","['https://openalex.org/T10036', 'https://opena...","['Advanced Neural Network Applications', 'Adva...","[0.9998999834060669, 0.9994999766349792, 0.999..."
4,https://openalex.org/W2117539524,https://doi.org/10.1007/s11263-015-0816-y,ImageNet Large Scale Visual Recognition Challenge,"[{'author_position': 'first', 'author': {'id':...","[{'id': 'https://openalex.org/T10627', 'displa...","{'id': 'https://openalex.org/T10627', 'display...",39595,2015,"['https://openalex.org/W2095705906', 'https://...","[{'id': 'https://openalex.org/C185798385', 'wi...","[{'author_position': 'first', 'author': {'id':...","['https://openalex.org/

In [14]:
df = df.groupby('title').first().reset_index()   # delete later - this was added to the preprocessing.py


In [20]:
def safe_parse(x):
    if isinstance(x, list):
        return x
    if isinstance(x, str):
        try:
            return ast.literal_eval(x)
        except:
            return []
    return []

df['author_names'] = df['author_names'].apply(safe_parse)
df['raw_affiliations'] = df['raw_affiliations'].apply(safe_parse)
df['author_ids'] = df['author_ids'].apply(safe_parse)


In [21]:
# Extract author-level edges using pre-parsed column
#MAX_AUTHORS = 20  # skip papers with too many authors

edge_weights = Counter()
author_info = {}

for row in df.itertuples(index=False):
    author_names = row.author_names
    raw_affiliations = row.raw_affiliations
    author_ids = row.author_ids

    if not author_names or len(author_names) < 2:
        continue

    if len(author_names) > MAX_AUTHORS:
        continue

    # remove duplicates just in case
    author_names = list(set(author_names))

    # store metadata
    for i, name in enumerate(author_names):
        if name not in author_info:
            author_info[name] = {
                'id': author_ids[i] if i < len(author_ids) else None,
                'affiliation': raw_affiliations[i] if i < len(raw_affiliations) else None
            }

    # generate edges
    for a, b in combinations(sorted(author_names), 2):
        edge_weights[(a, b)] += 1

In [ ]:
# Build NetworkX graph
G = nx.Graph()
for (auth1, auth2), weight in edge_weights.items():
    G.add_edge(auth1, auth2, weight=weight)

# Compute metrics
degree_centrality = nx.degree_centrality(G)
betweenness_centrality = nx.betweenness_centrality(G, weight='weight')
eigenvector_centrality = nx.eigenvector_centrality(G, weight='weight')


In [ ]:
# show graph
threshold = 3

H = nx.Graph(
    ((u, v, d) for u, v, d in G.edges(data=True) if d['weight'] >= threshold)
)

nx.draw(H, pos=nx.spring_layout(H), node_size=50)
plt.show()


str

In [ ]:
#print metrics
print(degree_centrality)
print(betweenness_centrality)
print(eigenvector_centrality)

{"'": 1.0170940170940173, '[': 1.0, 'M': 0.6068376068376069, 'o': 0.7094017094017094, 'n': 0.7606837606837608, 'i': 0.7179487179487181, 'q': 0.33333333333333337, 'u': 0.6837606837606838, 'e': 0.7948717948717949, ' ': 1.0170940170940173, 'B': 0.5726495726495727, 'r': 0.7435897435897436, 'd': 0.6581196581196582, 'a': 0.7606837606837608, 'g': 0.5726495726495727, ']': 1.0, 'J': 0.6153846153846154, 'K': 0.6324786324786326, 'm': 0.6581196581196582, 'D': 0.5897435897435898, 'v': 0.5384615384615385, 'H': 0.5470085470085471, 's': 0.6666666666666667, 'h': 0.6324786324786326, 'L': 0.5641025641025642, 't': 0.6666666666666667, 'A': 0.5897435897435898, 'l': 0.6495726495726496, 'G': 0.5384615384615385, 'b': 0.5982905982905984, '.': 0.7948717948717949, 'P': 0.5726495726495727, 'T': 0.5470085470085471, 'ö': 0.3931623931623932, 'S': 0.6153846153846154, 'p': 0.5384615384615385, 'c': 0.6068376068376069, 'E': 0.5811965811965812, 'W': 0.5213675213675214, 'k': 0.6153846153846154, 'w': 0.5042735042735044, 'R'

In [ ]:

# Save network metrics
metrics_df = pd.DataFrame({
    'author': list(G.nodes),
    'degree_centrality': [degree_centrality[a] for a in G.nodes],
    'betweenness_centrality': [betweenness_centrality[a] for a in G.nodes],
    'eigenvector_centrality': [eigenvector_centrality[a] for a in G.nodes],
    'id': [author_info[a]['id'] for a in G.nodes],
    'affiliation': [author_info[a]['affiliation'] for a in G.nodes]
})

#metrics_df.to_csv("../data/processed/author_network_metrics.csv", index=False)

In [ ]:
# Save full edge list
edges_df = pd.DataFrame([
    {'author_1': a1, 'author_2': a2, 'weight': w}
    for (a1, a2), w in edge_weights.items()
])
#edges_df.to_csv("../data/processed/author_network_edges.csv", index=False)

#print("Author network created and metrics saved!")